<a href="https://colab.research.google.com/github/MatthiasVanW/ISYS2001-Assessment-2-Programming-Assignment-Matthias-Vanwolleghem/blob/main/ISYS2001_Programming_Assignment_Matthias_Vanwolleghem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smart finance assistant: personal spending analyser
**Course**: ISYS2001 Introduction to Business Programming  
**Author**: Matthias Vanwolleghem  
**Project**: Assessment 2: Programming Assignment
____________________

## Six-Step Method

### Step 1: understanding the problem
When I look at my own spending reports, I notice they are often messy or hard to understand. This makes it frustrating to see where my money actually goes each month.

To solve this problem, I'm designing and building my own personal spending analyser. My goal is to create a simple tool that helps me to know where my money is going and how I can spend smarter.

To achieve this, my application will:
1. Safely load and clean raw CSV transaction files that I upload.
2. Run my own custom analytical logic to calculate category totals, spending percentages and identify my biggest expenses.
3. Include my own self-conceived feature that converts my spending totals into the equivalent number of beers (assuming $10 per beer).
4. Pass my clean financial numbers as context to a personal AI finance coach powered by Gemini, so it can give me direct data-driven advice instead of generic tips
5. Put a clear interactive Gradio interface so I can upload files and chat with my assistant without looking at the Python code.

----------

### Step 2: dentify inputs and outputs

* **Inputs**:
  * A CSV file I upload containing my transaction records ('Category', 'Amount' and 'Date')
  * Questions or prompts I type into the chat interface.

* **Outputs**:
  * A clean summary showing how much I spent per category.
  * Key figures like: my total spend, top spending category, spending percentages and my total spend expressed in number of beers.
  * Personalised data-driven financial advice from Gemini based on my actual numbers.

----------

### Step 3: worked example by hand

To test my logic I'll work throug a small fictive example.

Suppose I upload a CSV with these rows:
- `Date`: 2026-10-01, `Category`: Groceries, `Amount`: "\$50.00"
- `Date`: 2026-10-02, `Category`: Groceries, `Amount`: "\$30.00"
- `Date`: 2026-10-03, `Category`: Entertainment, `Amount`: "\$20.00"

1. **Clean**: I strip the `$` signs and change the values to float numbers: `[50.0, 30.0, 20.0]`.
2. **Group & aggregate**:
  - **Groceries**: \$50.00 + $30.00 = \$80.00 (80% of my spendings; equivalent to 2 beers at \$10 each)
  - **Entertainment**: \$20.00 (20% of my spend; equivalent to 8 beers at \$10 each)
3. **Total spend**: \$80.00 + \$20.00 = \$100.00
4. **Context for AI**: I feed this clean summary to Gemini: `"Total spent: $100.00 (10 beers). Top category: Groceries ($80.00 / 8 beers). How can I trim this?"`

----------

### Step 4: pseudocode

```text
CLASS Transaction:
    PROPERTIES: date, category, amount

CLASS FinanceManager:
    CONSTRUCTOR:
        INITIALISE empty transactions list
        SET BEER_PRICE = 10.00

    METHOD load_from_csv(file_input):
        TRY:
            LOAD CSV into DataFrame
            CLEAN column names (strip whitespace, standardise capitalisation)
            CHECK for required columns ('Category', 'Amount')
            IF missing:
                RETURN clear error message to user
            CLEAN 'Amount' column (strip '$', convert to float)
            CONVERT valid rows into Transaction objects
            RETURN success message
        EXCEPT Exception as e:
            RETURN clear user error message

    METHOD analyze_spending():
        IF transactions list is empty:
            RETURN warning message
        CONVERT transactions to DataFrame
        GROUP BY 'Category' and SUM 'Amount'
        CALCULATE total spend and percentage per category
        CALCULATE beer_equivalent = total_spend / 10.00
        IDENTIFY top category and highest spend
        RETURN summary text (including beer metric) and formatted summary DataFrame

FUNCTION ask_financial_coach(user_query, spending_summary):
    CONFIGURE Gemini API with system instruction ("You are an empathetic financial coach...")
    COMBINE spending_summary + user_query into full prompt
    CALL Gemini API (gemini-3.8-flash)
    RETURN AI response
```

----------


### Gemini API connection test
(This setup and verification code was generated with AI assistance.)

I want to verify that the Google Gemini API key is properly configured and accessible via Google Colab Secrets (`GEMINI_API_KEY`), before I run my code.

The code block below retrieves the secure key using `google.colab.userdata` and performs a minimal call to the `gemini-3.8-flash` model to confirm successful authentication.

In [1]:
# Here I just asked AI to help me because I wanted to be sure that my Key was correctly installed

# 1. Benodigde bibliotheken installeren en importeren
!pip install -q google-genai

import os
from google import genai
from google.colab import userdata

# 2. De API-sleutel ophalen uit Colab Secrets
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)

    # 3. Testvraag sturen naar gemini-3.8-flash (of gemini-flash-latest)
    response = client.models.generate_content(
        model='gemini-3.8-flash',
        contents='Hallo, bevestig in één korte zin dat je de verbinding met het ISYS2001-project hebt gerealiseerd.'
    )

    print(" SUCCESS: Gemini API connection established!")
    print("Response from Gemini:\n", response.text)

except Exception as e:
    print(" ERROR: Could not connect to Gemini API.")
    print("Details:", e)

 SUCCESS: Gemini API connection established!
Response from Gemini:
 Ik bevestig dat de verbinding met het ISYS2001-project succesvol is gerealiseerd.



### Step 5: convert pseudocode to code


In [11]:
class Transaction:
    """Represents an individual financial transaction."""

    def __init__(self, date: str, category: str, amount: float, description: str = ""):
#store essential details for a single expense
        self.date = date
        self.category = category
        self.amount = amount
        self.description = description

    def __repr__(self):
#clean formatting for displaying object information
        return f"Transaction({self.date}, {self.category}, ${self.amount:.2f})"

import pandas as pd

class FinanceManager:
    """Manages transaction records, data cleaning, and custom analytics."""

    def __init__(self, beer_price: float = 10.0):
#list to hold Transaction objects and standard beer conversion cost
        self.transactions = []
        self.beer_price = beer_price

    def load_from_dataframe(self, df: pd.DataFrame):
        """Cleans raw DataFrame input and converts rows into Transaction objects."""
#standardise column headers (remove whitespaces and lowercase)
        df.columns = df.columns.str.strip().str.lower()

#check for mandatory columns
        required_cols = {'date', 'category', 'amount'}
        if not required_cols.issubset(df.columns):
            raise ValueError(f"CSV must contain columns: {required_cols}")

        self.transactions = []
        for _, row in df.iterrows():
#clean currency symbols ($) and convert text to float
            raw_amount = str(row['amount']).replace('$', '').strip()
            try:
                clean_amount = float(raw_amount)
            except ValueError:
                continue

            desc = row.get('description', '')
#create a Transaction instance per row
            t = Transaction(
                date=str(row['date']),
                category=str(row['category']).strip().title(),
                amount=clean_amount,
                description=str(desc)
            )
            self.transactions.append(t)

    def calculate_totals_by_category(self) -> dict:
        """Aggregates total spending grouped by category."""
        category_totals = {}
#sum up amounts per unique category
        for t in self.transactions:
            category_totals[t.category] = category_totals.get(t.category, 0.0) + t.amount
        return category_totals

    def get_summary_report(self) -> dict:
        """Generates comprehensive expenditure summary and custom analytics."""
#safety check for empty transaction list
        if not self.transactions:
            return {"error": "No valid transactions loaded."}

        total_spent = sum(t.amount for t in self.transactions)
        cat_totals = self.calculate_totals_by_category()

#calculate percentages per cat
        cat_percentages = {
            cat: round((amt / total_spent) * 100, 1)
            for cat, amt in cat_totals.items()
        } if total_spent > 0 else {}

#identifie top 3 largest expenses
        sorted_transactions = sorted(self.transactions, key=lambda t: t.amount, reverse=True)
        top_expenses = [
            {"date": t.date, "category": t.category, "amount": t.amount, "description": t.description}
            for t in sorted_transactions[:3]
        ]
#custom beer converter
        beers_equivalent = total_spent / self.beer_price if self.beer_price > 0 else 0

        return {
            "total_spending": round(total_spent, 2),
            "category_breakdown": {cat: round(amt, 2) for cat, amt in cat_totals.items()},
            "category_percentages": cat_percentages,
            "top_expenses": top_expenses,
            "total_beers_equivalent": round(beers_equivalent, 1),
            "transaction_count": len(self.transactions)
        }

Now I'll test the `Transaction` and `FinanceManager` classes using a small made up sample dataset to verify that everything works as expected.

In this test I check:
- Whether the `$` symbols are properly removed and amounts are converted into numeric values.
- Whether category breakdowns, percentages, top expenses and the beer converter gets calculated correctly.
- Whether the Gemini API reads the summary properly and returns useful budget coaching

In [17]:
!pip install -q google-generativeai

import pandas as pd
from google.colab import userdata
import google.generativeai as genai

#create sample test data with messy input
test_data = {
    'Date': ['2026-10-01', '2026-10-02', '2026-10-03', '2026-10-04'],
    'Category': [' Groceries ', 'Entertainment', 'Groceries', 'Tech'],
    'Amount': ['$45.50', '20.00', '$34.50', '$120.00'],
    'Description': ['Weekly food', 'Movie ticket', 'Snacks', 'New Headphones']
}
df_test = pd.DataFrame(test_data)

#test FinanceManager analytics
manager = FinanceManager(beer_price=10.0)
manager.load_from_dataframe(df_test)
summary = manager.get_summary_report()

print("--- ANALYTICS REPORT ---")
print(f"Total Spent: ${summary['total_spending']}")
print(f"Category Percentages: {summary['category_percentages']}")
print(f"Top Expense: {summary['top_expenses'][0]}")
print(f"Beer Equivalent ($10/beer): {summary['total_beers_equivalent']} beers")


#test Gemini API integration (this part is completely written by AI)
api_key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=api_key)

model = genai.GenerativeModel('gemini-3.8-flash')
prompt = f"""
You are a friendly and constructive AI Financial Coach.
Analyse this spending summary:
- Total Spent: ${summary['total_spending']}
- Category Percentages: {summary['category_percentages']}
- Top Expenses: {summary['top_expenses']}
- Total Beers Equivalent ($10/beer): {summary['total_beers_equivalent']} beers

Provide 2 short budgeting tips. Include a lighthearted reference to their spending converted into beers.
"""

response = model.generate_content(prompt)
print("\n--- GEMINI AI COACH ADVICE ---")
print(response.text)

--- ANALYTICS REPORT ---
Total Spent: $220.0
Category Percentages: {'Groceries': 36.4, 'Entertainment': 9.1, 'Tech': 54.5}
Top Expense: {'date': '2026-10-04', 'category': 'Tech', 'amount': 120.0, 'description': 'New Headphones'}
Beer Equivalent ($10/beer): 22.0 beers

--- GEMINI AI COACH ADVICE ---
Hey there! Overall, you’ve kept your spending relatively lean at **$220**, though to put it in weekend terms, that’s about **22 pub beers**! 🍻 

Tech took the lion’s share this round at **54.5%** thanks to those new headphones, while your day-to-day groceries sat nicely at **36.4%**. 

Here are two quick tips to keep your budget cruising smoothly:

1. **Set Up a "Tech Sinking Fund":** One-off gadgets like your $120 headphones can heavily tip your category percentages. Stashing away $15–$20 a week into a dedicated tech fund lets you treat yourself to upgrades completely guilt-free, without feeling the pinch all at once.
2. **Watch the "Sneaky Snack" Tax:** Your snack run ($34.50) was almost a

### Step 6: test

(later)